# 05 - Data analysis

Respuestas a las 20 preguntas del enunciado usando `ANALYTICS.OBT_TRIPS`.

In [1]:
import os
from pyspark.sql import SparkSession


In [2]:
spark_snowflake_pkg = os.getenv('SPARK_SNOWFLAKE_PACKAGE', 'net.snowflake:spark-snowflake_2.12:3.1.8')
snowflake_jdbc_pkg = os.getenv('SNOWFLAKE_JDBC_PACKAGE', 'net.snowflake:snowflake-jdbc:3.14.4')

spark = (
    SparkSession.builder
    .appName('PSet3 Data Analysis')
    .master('local[*]')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.jars.packages', f"{spark_snowflake_pkg},{snowflake_jdbc_pkg}")
    .config('spark.sql.session.timeZone', 'UTC')
    .getOrCreate()
)

print(spark.version)

3.5.0


In [3]:
def get_env(name, default=None, required=False):
    value = os.getenv(name, default)
    if required and (value is None or str(value).strip() == ''):
        raise ValueError(f'Falta variable de entorno requerida: {name}')
    return value

snowflake_host = get_env('SNOWFLAKE_HOST', '').strip()
snowflake_account = get_env('SNOWFLAKE_ACCOUNT', '').strip()
snowflake_port = get_env('SNOWFLAKE_PORT', '443').strip()

if not snowflake_host:
    if not snowflake_account:
        raise ValueError('Debes configurar SNOWFLAKE_HOST o SNOWFLAKE_ACCOUNT')
    snowflake_host = f"{snowflake_account}.snowflakecomputing.com"

if snowflake_port and snowflake_port != '443':
    snowflake_host = f"{snowflake_host}:{snowflake_port}"

sf_options = {
    'sfURL': snowflake_host,
    'sfUser': get_env('SNOWFLAKE_USER', required=True),
    'sfPassword': get_env('SNOWFLAKE_PASSWORD', required=True),
    'sfRole': get_env('SNOWFLAKE_ROLE', required=True),
    'sfWarehouse': get_env('SNOWFLAKE_WAREHOUSE', required=True),
    'sfDatabase': get_env('SNOWFLAKE_DATABASE', required=True),
    'sfSchema': get_env('SNOWFLAKE_SCHEMA_ANALYTICS', 'ANALYTICS'),
}

analytics_schema = get_env('SNOWFLAKE_SCHEMA_ANALYTICS', 'ANALYTICS')
obt_table_name = f"{analytics_schema}.OBT_TRIPS"
obt_table = f"""(
  SELECT *
  FROM {obt_table_name}
  WHERE year BETWEEN 2015 AND 2025
    AND pickup_datetime >= TO_TIMESTAMP('2015-01-01 00:00:00')
    AND pickup_datetime < TO_TIMESTAMP('2026-01-01 00:00:00')
    AND dropoff_datetime >= TO_TIMESTAMP('2015-01-01 00:00:00')
    AND dropoff_datetime < TO_TIMESTAMP('2026-01-01 00:00:00')
    AND ABS(year - TRY_TO_NUMBER(source_year)) <= 1
    AND trip_distance BETWEEN 0.1 AND 100
    AND trip_duration_min BETWEEN 1 AND 240
    AND (avg_speed_mph IS NULL OR avg_speed_mph BETWEEN 1 AND 120)
    AND (fare_amount IS NULL OR fare_amount >= 0)
    AND (total_amount IS NULL OR total_amount >= 0)
    AND (passenger_count IS NULL OR passenger_count BETWEEN 1 AND 6)
)"""

{k: ('***' if 'Password' in k else v) for k, v in sf_options.items()}

{'sfURL': 'DVNRQOK-CRC15844.snowflakecomputing.com',
 'sfUser': 'tonyfajardo1d',
 'sfPassword': '***',
 'sfRole': 'ACCOUNTADMIN',
 'sfWarehouse': 'PSET3_WH',
 'sfDatabase': 'DM_PSET3',
 'sfSchema': 'ANALYTICS'}

## a) Top 10 zonas de pickup por volumen mensual

Interpretacion: las zonas lideres de pickup son consistentemente de Manhattan (Upper East Side, Midtown, Union Sq), lo que confirma concentracion estructural de demanda en esas areas.

In [4]:
q_a = f"""
WITH base AS (
  SELECT year, month, pu_zone, COUNT(*) AS trips
  FROM {obt_table}
  GROUP BY 1,2,3
), ranked AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY year, month ORDER BY trips DESC) AS rn
  FROM base
)
SELECT year, month, pu_zone, trips
FROM ranked
WHERE rn <= 10
ORDER BY year, month, trips DESC
"""
spark.read.format('snowflake').options(**sf_options).option('query', q_a).load().show(120, truncate=False)

+----+-----+----------------------------+------+
|YEAR|MONTH|PU_ZONE                     |TRIPS |
+----+-----+----------------------------+------+
|2015|1    |Upper East Side South       |463868|
|2015|1    |Midtown Center              |443068|
|2015|1    |Upper East Side North       |441747|
|2015|1    |East Village                |434248|
|2015|1    |Times Sq/Theatre District   |425407|
|2015|1    |Union Sq                    |420289|
|2015|1    |Murray Hill                 |407996|
|2015|1    |Midtown East                |407534|
|2015|1    |Clinton East                |397346|
|2015|1    |Penn Station/Madison Sq West|389978|
|2015|2    |Upper East Side South       |443896|
|2015|2    |East Village                |418379|
|2015|2    |Midtown Center              |415053|
|2015|2    |Upper East Side North       |413821|
|2015|2    |Union Sq                    |412561|
|2015|2    |Times Sq/Theatre District   |402399|
|2015|2    |Midtown East                |401696|
|2015|2    |Murray H

## b) Top 10 zonas de dropoff por volumen mensual

Interpretacion: los dropoffs top siguen corredores similares a los pickups, indicando alta recurrencia de viajes intra-Manhattan y entre zonas de alta actividad.

In [5]:
q_b = f"""
WITH base AS (
  SELECT year, month, do_zone, COUNT(*) AS trips
  FROM {obt_table}
  GROUP BY 1,2,3
), ranked AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY year, month ORDER BY trips DESC) AS rn
  FROM base
)
SELECT year, month, do_zone, trips
FROM ranked
WHERE rn <= 10
ORDER BY year, month, trips DESC
"""
spark.read.format('snowflake').options(**sf_options).option('query', q_b).load().show(120, truncate=False)

+----+-----+----------------------------+------+
|YEAR|MONTH|DO_ZONE                     |TRIPS |
+----+-----+----------------------------+------+
|2015|1    |Midtown Center              |470029|
|2015|1    |Upper East Side North       |454015|
|2015|1    |Upper East Side South       |408500|
|2015|1    |Murray Hill                 |404171|
|2015|1    |Times Sq/Theatre District   |398924|
|2015|1    |Midtown East                |377253|
|2015|1    |Union Sq                    |368533|
|2015|1    |East Village                |368253|
|2015|1    |Clinton East                |353579|
|2015|1    |Penn Station/Madison Sq West|344034|
|2015|2    |Midtown Center              |459326|
|2015|2    |Upper East Side North       |421556|
|2015|2    |Murray Hill                 |389265|
|2015|2    |Upper East Side South       |387731|
|2015|2    |Times Sq/Theatre District   |373187|
|2015|2    |Midtown East                |369606|
|2015|2    |Union Sq                    |366072|
|2015|2    |East Vil

## c) Evolucion mensual de total_amount y tip_pct por borough

Interpretacion: boroughs con mayor volumen muestran mayor total_amount agregado; el tip_pct promedio es mas alto donde predomina pago con tarjeta y viajes de mayor ticket.

In [6]:
q_c = f"""
SELECT year, month, pu_borough,
       ROUND(SUM(total_amount), 2) AS total_amount_sum,
       ROUND(AVG(tip_pct), 4) AS avg_tip_pct
FROM {obt_table}
GROUP BY 1,2,3
ORDER BY 1,2,3
"""
spark.read.format('snowflake').options(**sf_options).option('query', q_c).load().show(120, truncate=False)

+----+-----+-------------+----------------+-----------+
|YEAR|MONTH|PU_BOROUGH   |TOTAL_AMOUNT_SUM|AVG_TIP_PCT|
+----+-----+-------------+----------------+-----------+
|2015|1    |Bronx        |1201815.25      |0.0297     |
|2015|1    |Brooklyn     |1.229854734E7   |0.124      |
|2015|1    |EWR          |5073.78         |0.1286     |
|2015|1    |Manhattan    |1.6438354132E8  |0.1453     |
|2015|1    |N/A          |117684.21       |1.8954     |
|2015|1    |Queens       |3.036351107E7   |0.1242     |
|2015|1    |Staten Island|9550.34         |0.0901     |
|2015|1    |Unknown      |3268779.69      |0.1314     |
|2015|2    |Bronx        |1630670.74      |0.0251     |
|2015|2    |Brooklyn     |1.322789093E7   |0.128      |
|2015|2    |EWR          |4588.32         |0.1171     |
|2015|2    |Manhattan    |1.6295845676E8  |0.1365     |
|2015|2    |N/A          |132698.99       |1.2282     |
|2015|2    |Queens       |2.955120262E7   |0.1189     |
|2015|2    |Staten Island|8874.73         |0.080

## d) Ticket promedio por service_type y mes

Interpretacion: yellow mantiene escala de volumen superior y green muestra variaciones de ticket medio mas sensibles a estacionalidad y mezcla de zonas.

In [7]:
q_d = f"""
SELECT year, month, service_type, ROUND(AVG(total_amount), 2) AS avg_ticket
FROM {obt_table}
GROUP BY 1,2,3
ORDER BY 1,2,3
"""
spark.read.format('snowflake').options(**sf_options).option('query', q_d).load().show(120, truncate=False)

+----+-----+------------+----------+
|YEAR|MONTH|SERVICE_TYPE|AVG_TICKET|
+----+-----+------------+----------+
|2015|1    |green       |14.76     |
|2015|1    |yellow      |15.07     |
|2015|2    |green       |14.41     |
|2015|2    |yellow      |15.34     |
|2015|3    |green       |14.49     |
|2015|3    |yellow      |15.81     |
|2015|4    |green       |14.75     |
|2015|4    |yellow      |15.99     |
|2015|5    |green       |15.19     |
|2015|5    |yellow      |16.38     |
|2015|6    |green       |14.89     |
|2015|6    |yellow      |16.34     |
|2015|7    |green       |14.88     |
|2015|7    |yellow      |16.09     |
|2015|8    |green       |14.93     |
|2015|8    |yellow      |16.1      |
|2015|9    |green       |15.07     |
|2015|9    |yellow      |16.41     |
|2015|10   |green       |14.92     |
|2015|10   |yellow      |16.48     |
|2015|11   |green       |14.71     |
|2015|11   |yellow      |16.31     |
|2015|12   |green       |14.7      |
|2015|12   |yellow      |16.24     |
|

## e) Viajes por hora del dia y dia de semana

Interpretacion: los picos se concentran en horas laborales y ventanas de desplazamiento diario, evidenciando patrones de movilidad urbana recurrentes.

In [8]:
q_e = f"""
SELECT pickup_hour, day_of_week, COUNT(*) AS trips
FROM {obt_table}
GROUP BY 1,2
ORDER BY trips DESC
"""
spark.read.format('snowflake').options(**sf_options).option('query', q_e).load().show(120, truncate=False)

+-----------+-----------+-------+
|PICKUP_HOUR|DAY_OF_WEEK|TRIPS  |
+-----------+-----------+-------+
|18         |5          |8335194|
|18         |6          |8306393|
|19         |6          |8298294|
|18         |4          |8249073|
|19         |5          |8139978|
|18         |3          |8126472|
|19         |4          |8007353|
|21         |5          |7841686|
|20         |5          |7783223|
|19         |7          |7754601|
|18         |7          |7683038|
|19         |3          |7641274|
|21         |4          |7588406|
|20         |4          |7561180|
|18         |2          |7487443|
|22         |6          |7449029|
|22         |5          |7378101|
|20         |6          |7311051|
|17         |6          |7278819|
|17         |5          |7269441|
|20         |3          |7234739|
|17         |4          |7227410|
|21         |3          |7224711|
|23         |7          |7202942|
|23         |6          |7186526|
|21         |6          |7169044|
|22         |7

## f) p50/p90 de trip_duration_min por borough

Interpretacion: Manhattan presenta duraciones medianas mas bajas, mientras Queens/Staten Island y zonas especiales tienen p90 mayores por distancias y condiciones de trafico.

In [9]:
q_f = f"""
SELECT pu_borough,
       PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY trip_duration_min) AS p50_duration,
       PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY trip_duration_min) AS p90_duration
FROM {obt_table}
GROUP BY 1
ORDER BY 1
"""
spark.read.format('snowflake').options(**sf_options).option('query', q_f).load().show(truncate=False)

+-------------+------------------+------------------+
|PU_BOROUGH   |P50_DURATION      |P90_DURATION      |
+-------------+------------------+------------------+
|Bronx        |14.166666666666666|39.0              |
|Brooklyn     |13.183333333333334|32.63333333333333 |
|EWR          |10.716666666666667|57.04500000000002 |
|Manhattan    |10.9              |25.066666666666666|
|N/A          |38.75             |60.43333333333333 |
|Queens       |25.283333333333335|54.63333333333333 |
|Staten Island|30.883333333333333|72.35             |
|Unknown      |11.433333333333334|28.933333333333334|
+-------------+------------------+------------------+



## g) avg_speed_mph por franja horaria (6-9, 17-20) y borough

Interpretacion: en franjas pico, Manhattan refleja menores velocidades relativas que boroughs perifericos, consistente con mayor congestion en zonas centrales.

In [10]:
q_g = f"""
SELECT
  CASE
    WHEN pickup_hour BETWEEN 6 AND 9 THEN '06-09'
    WHEN pickup_hour BETWEEN 17 AND 20 THEN '17-20'
    ELSE 'other'
  END AS time_band,
  pu_borough,
  ROUND(AVG(avg_speed_mph), 2) AS avg_speed_mph
FROM {obt_table}
WHERE pickup_hour BETWEEN 6 AND 9 OR pickup_hour BETWEEN 17 AND 20
GROUP BY 1,2
ORDER BY 1,2
"""
spark.read.format('snowflake').options(**sf_options).option('query', q_g).load().show(120, truncate=False)

+---------+-------------+-------------+
|TIME_BAND|PU_BOROUGH   |AVG_SPEED_MPH|
+---------+-------------+-------------+
|06-09    |Bronx        |14.37        |
|06-09    |Brooklyn     |13.25        |
|06-09    |EWR          |23.29        |
|06-09    |Manhattan    |11.43        |
|06-09    |N/A          |14.62        |
|06-09    |Queens       |18.39        |
|06-09    |Staten Island|20.6         |
|06-09    |Unknown      |11.97        |
|17-20    |Bronx        |13.04        |
|17-20    |Brooklyn     |11.35        |
|17-20    |EWR          |23.69        |
|17-20    |Manhattan    |9.99         |
|17-20    |N/A          |14.89        |
|17-20    |Queens       |18.85        |
|17-20    |Staten Island|21.81        |
|17-20    |Unknown      |10.9         |
+---------+-------------+-------------+



## h) Participacion por payment_type_desc y relacion con tip_pct

Interpretacion: tarjeta domina participacion y concentra tip_pct mas alto; cash mantiene propina porcentual cercana a cero como comportamiento esperado.

In [11]:
q_h = f"""
SELECT payment_type_desc,
       COUNT(*) AS trips,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS share_pct,
       ROUND(AVG(tip_pct), 4) AS avg_tip_pct
FROM {obt_table}
GROUP BY 1
ORDER BY trips DESC
"""
spark.read.format('snowflake').options(**sf_options).option('query', q_h).load().show(truncate=False)

+-----------------+---------+---------+-----------+
|PAYMENT_TYPE_DESC|TRIPS    |SHARE_PCT|AVG_TIP_PCT|
+-----------------+---------+---------+-----------+
|Credit card      |570629400|67.62    |0.2336     |
|Cash             |249682019|29.59    |0.0        |
|Unknown          |19102700 |2.26     |0.0543     |
|No charge        |2540293  |0.30     |5.0E-4     |
|Dispute          |1941953  |0.23     |6.0E-4     |
+-----------------+---------+---------+-----------+



## i) Rate codes con mayor trip_distance y total_amount

Interpretacion: Standard rate lidera monto total por volumen, mientras codigos de aeropuerto y especiales muestran mayor distancia promedio por viaje.

In [12]:
q_i = f"""
SELECT rate_code_desc,
       ROUND(AVG(trip_distance), 2) AS avg_trip_distance,
       ROUND(SUM(total_amount), 2) AS sum_total_amount
FROM {obt_table}
GROUP BY 1
ORDER BY sum_total_amount DESC, avg_trip_distance DESC
"""
spark.read.format('snowflake').options(**sf_options).option('query', q_i).load().show(truncate=False)

+---------------------+-----------------+----------------+
|RATE_CODE_DESC       |AVG_TRIP_DISTANCE|SUM_TOTAL_AMOUNT|
+---------------------+-----------------+----------------+
|Standard rate        |2.64             |1.34038441536E10|
|JFK                  |18.0             |1.35206206178E9 |
|Unknown              |4.92             |5.9807047391E8  |
|Newark               |17.33            |1.607252636E8   |
|Negotiated fare      |9.9              |1.6031665634E8  |
|Nassau or Westchester|19.5             |7.374169121E7   |
|Group ride           |2.68             |41357.26        |
+---------------------+-----------------+----------------+



## j) Mix yellow vs green por mes y borough

Interpretacion: yellow predomina fuertemente en Manhattan; green gana peso relativo en Brooklyn/Bronx, mostrando especializacion por territorio y tipo de servicio.

In [13]:
q_j = f"""
SELECT year, month, pu_borough, service_type, COUNT(*) AS trips
FROM {obt_table}
GROUP BY 1,2,3,4
ORDER BY 1,2,3,4
"""
spark.read.format('snowflake').options(**sf_options).option('query', q_j).load().show(150, truncate=False)

+----+-----+-------------+------------+--------+
|YEAR|MONTH|PU_BOROUGH   |SERVICE_TYPE|TRIPS   |
+----+-----+-------------+------------+--------+
|2015|1    |Bronx        |green       |84190   |
|2015|1    |Bronx        |yellow      |8761    |
|2015|1    |Brooklyn     |green       |560029  |
|2015|1    |Brooklyn     |yellow      |225394  |
|2015|1    |EWR          |green       |5       |
|2015|1    |EWR          |yellow      |84      |
|2015|1    |Manhattan    |green       |421832  |
|2015|1    |Manhattan    |yellow      |11520258|
|2015|1    |N/A          |green       |320     |
|2015|1    |N/A          |yellow      |3756    |
|2015|1    |Queens       |green       |400690  |
|2015|1    |Queens       |yellow      |620407  |
|2015|1    |Staten Island|green       |173     |
|2015|1    |Staten Island|yellow      |142     |
|2015|1    |Unknown      |green       |1233    |
|2015|1    |Unknown      |yellow      |223176  |
|2015|2    |Bronx        |green       |114626  |
|2015|2    |Bronx   

## k) Top 20 flujos PU->DO por volumen y ticket promedio

Interpretacion: los flujos top son mayormente intra-zona o entre zonas vecinas de alta densidad, con tickets promedio estables y alto volumen repetitivo.

In [14]:
q_k = f"""
SELECT pu_zone, do_zone, COUNT(*) AS trips, ROUND(AVG(total_amount), 2) AS avg_ticket
FROM {obt_table}
GROUP BY 1,2
ORDER BY trips DESC
LIMIT 20
"""
spark.read.format('snowflake').options(**sf_options).option('query', q_k).load().show(truncate=False)

+----------------------------+----------------------------+-------+----------+
|PU_ZONE                     |DO_ZONE                     |TRIPS  |AVG_TICKET|
+----------------------------+----------------------------+-------+----------+
|N/A                         |N/A                         |7076314|16.61     |
|Upper East Side South       |Upper East Side North       |4517488|10.48     |
|Upper East Side North       |Upper East Side South       |3860652|11.28     |
|Upper East Side North       |Upper East Side North       |3430923|8.52      |
|Upper East Side South       |Upper East Side South       |3292217|9.08      |
|Upper West Side South       |Upper West Side North       |2001720|9.01      |
|Upper West Side South       |Lincoln Square East         |1988033|9.55      |
|Upper East Side South       |Midtown Center              |1935949|12.31     |
|Upper East Side South       |Midtown East                |1922458|10.87     |
|Lincoln Square East         |Upper West Side South 

## l) Distribucion de passenger_count y efecto en total_amount

Interpretacion: passenger_count=1 domina ampliamente; grupos mayores son minoritarios y suelen asociarse a tickets medios distintos por tipo de recorrido.

In [15]:
q_l = f"""
SELECT passenger_count, COUNT(*) AS trips, ROUND(AVG(total_amount), 2) AS avg_total_amount
FROM {obt_table}
GROUP BY 1
ORDER BY trips DESC
"""
spark.read.format('snowflake').options(**sf_options).option('query', q_l).load().show(truncate=False)

+---------------+---------+----------------+
|PASSENGER_COUNT|TRIPS    |AVG_TOTAL_AMOUNT|
+---------------+---------+----------------+
|1.0            |605906560|18.26           |
|2.0            |117232257|19.63           |
|5.0            |33143193 |17.06           |
|3.0            |32444878 |19.04           |
|6.0            |20388012 |16.9            |
|NULL           |19100685 |28.42           |
|4.0            |15680780 |19.92           |
+---------------+---------+----------------+



## m) Impacto de tolls_amount y congestion_surcharge por zona

Interpretacion: zonas con mayor surcharge/peajes muestran incremento de total_amount, confirmando efecto directo de recargos en el ticket final.

In [16]:
q_m = f"""
SELECT pu_zone,
       ROUND(AVG(tolls_amount), 2) AS avg_tolls,
       ROUND(AVG(congestion_surcharge), 2) AS avg_congestion,
       ROUND(AVG(total_amount), 2) AS avg_total_amount
FROM {obt_table}
GROUP BY 1
ORDER BY avg_congestion DESC, avg_tolls DESC
LIMIT 30
"""
spark.read.format('snowflake').options(**sf_options).option('query', q_m).load().show(truncate=False)

+-----------------------------+---------+--------------+----------------+
|PU_ZONE                      |AVG_TOLLS|AVG_CONGESTION|AVG_TOTAL_AMOUNT|
+-----------------------------+---------+--------------+----------------+
|Battery Park                 |0.2      |2.47          |23.1            |
|West Village                 |0.09     |2.47          |16.46           |
|Upper East Side South        |0.07     |2.47          |14.36           |
|Greenwich Village South      |0.06     |2.47          |16.46           |
|Midtown Center               |0.28     |2.46          |17.51           |
|Midtown East                 |0.26     |2.46          |17.01           |
|East Chelsea                 |0.2      |2.46          |17.11           |
|Penn Station/Madison Sq West |0.19     |2.46          |17.06           |
|Yorkville East               |0.19     |2.46          |15.99           |
|Gramercy                     |0.17     |2.46          |15.61           |
|West Chelsea/Hudson Yards    |0.16   

## n) Proporcion de viajes cortos vs largos por borough y estacionalidad

Interpretacion: viajes cortos predominan en zonas densas; viajes largos ganan presencia relativa en boroughs perifericos y cambian por estacionalidad mensual.

In [17]:
q_n = f"""
SELECT year, month, pu_borough,
       SUM(CASE WHEN trip_distance < 2 THEN 1 ELSE 0 END) AS short_trips,
       SUM(CASE WHEN trip_distance >= 10 THEN 1 ELSE 0 END) AS long_trips,
       COUNT(*) AS total_trips
FROM {obt_table}
GROUP BY 1,2,3
ORDER BY 1,2,3
"""
spark.read.format('snowflake').options(**sf_options).option('query', q_n).load().show(150, truncate=False)

+----+-----+-------------+-----------+----------+-----------+
|YEAR|MONTH|PU_BOROUGH   |SHORT_TRIPS|LONG_TRIPS|TOTAL_TRIPS|
+----+-----+-------------+-----------+----------+-----------+
|2015|1    |Bronx        |47109      |3173      |92951      |
|2015|1    |Brooklyn     |345022     |25987     |785423     |
|2015|1    |EWR          |53         |27        |89         |
|2015|1    |Manhattan    |7235265    |264507    |11942090   |
|2015|1    |N/A          |2193       |417       |4076       |
|2015|1    |Queens       |291171     |328836    |1021097    |
|2015|1    |Staten Island|89         |82        |315        |
|2015|1    |Unknown      |130979     |9994      |224409     |
|2015|2    |Bronx        |64031      |3790      |125245     |
|2015|2    |Brooklyn     |358918     |28145     |820401     |
|2015|2    |EWR          |49         |21        |79         |
|2015|2    |Manhattan    |7025089    |276361    |11656209   |
|2015|2    |N/A          |2611       |409       |4694       |
|2015|2 

## o) Diferencias por vendor en avg_speed_mph y trip_duration_min

Interpretacion: hay diferencias moderadas entre vendors, explicables por mezcla de zonas/horas atendidas y no necesariamente por desempeno operacional aislado.

In [18]:
q_o = f"""
SELECT vendor_name,
       ROUND(AVG(avg_speed_mph), 2) AS avg_speed_mph,
       ROUND(AVG(trip_duration_min), 2) AS avg_trip_duration_min,
       COUNT(*) AS trips
FROM {obt_table}
GROUP BY 1
ORDER BY trips DESC
"""
spark.read.format('snowflake').options(**sf_options).option('query', q_o).load().show(truncate=False)

+----------------------------+-------------+---------------------+---------+
|VENDOR_NAME                 |AVG_SPEED_MPH|AVG_TRIP_DURATION_MIN|TRIPS    |
+----------------------------+-------------+---------------------+---------+
|VeriFone Inc.               |11.84        |14.91                |528221272|
|Creative Mobile Technologies|11.3         |14.49                |314750667|
|Unknown                     |10.7         |21.99                |924426   |
+----------------------------+-------------+---------------------+---------+



## p) Relacion metodo de pago vs tip_amount por hora

Interpretacion: en casi todas las horas, tarjeta mantiene mayor tip_amount y tip_pct que efectivo, con estabilidad del patron a lo largo del dia.

In [19]:
q_p = f"""
SELECT pickup_hour, payment_type_desc,
       ROUND(AVG(tip_amount), 2) AS avg_tip_amount,
       ROUND(AVG(tip_pct), 4) AS avg_tip_pct,
       COUNT(*) AS trips
FROM {obt_table}
GROUP BY 1,2
ORDER BY 1, trips DESC
"""
spark.read.format('snowflake').options(**sf_options).option('query', q_p).load().show(200, truncate=False)

+-----------+-----------------+--------------+-----------+--------+
|PICKUP_HOUR|PAYMENT_TYPE_DESC|AVG_TIP_AMOUNT|AVG_TIP_PCT|TRIPS   |
+-----------+-----------------+--------------+-----------+--------+
|0          |Credit card      |3.01          |0.2377     |18781452|
|0          |Cash             |0.0           |0.0        |8031090 |
|0          |Unknown          |0.71          |0.0363     |794252  |
|0          |No charge        |0.0           |2.0E-4     |102730  |
|0          |Dispute          |0.01          |7.0E-4     |73483   |
|1          |Credit card      |2.78          |0.2411     |13213587|
|1          |Cash             |0.0           |0.0        |5883177 |
|1          |Unknown          |0.66          |0.034      |527226  |
|1          |No charge        |0.0           |4.0E-4     |83423   |
|1          |Dispute          |0.01          |4.0E-4     |54156   |
|2          |Credit card      |2.64          |0.24       |9261392 |
|2          |Cash             |0.0           |0.

## q) Zonas con percentil 99 de duracion/distancia fuera de rango

Interpretacion: los outliers p99 se concentran en periferia/aeropuertos y zonas de baja densidad, utiles para monitorear congestion excepcional o eventos.

In [20]:
q_q = f"""
SELECT pu_zone,
       PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY trip_duration_min) AS p99_duration,
       PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY trip_distance) AS p99_distance
FROM {obt_table}
GROUP BY 1
HAVING PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY trip_duration_min) > 180
    OR PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY trip_distance) > 30
ORDER BY p99_duration DESC, p99_distance DESC
LIMIT 50
"""
spark.read.format('snowflake').options(**sf_options).option('query', q_q).load().show(truncate=False)

+---------------------------------+------------------+------------------+
|PU_ZONE                          |P99_DURATION      |P99_DISTANCE      |
+---------------------------------+------------------+------------------+
|Arden Heights                    |165.27000000000015|41.898000000000025|
|Rossville/Woodrow                |137.99666666666644|47.40119999999998 |
|Far Rockaway                     |136.84499999999997|30.75349999999999 |
|Hammels/Arverne                  |133.91450000000003|31.634800000000105|
|Eltingville/Annadale/Prince's Bay|133.67699999999994|48.170599999999965|
|Rockaway Park                    |127.91916666666654|31.39             |
|Heartland Village/Todt Hill      |123.99483333333336|39.435500000000005|
|Outside of NYC                   |120.21666666666667|34.18             |
|Saint George/New Brighton        |118.47249999999977|34.44             |
|Charleston/Tottenville           |115.44949999999992|53.32789999999999 |
|Cambria Heights                  |113

## r) Yield por milla (total_amount/trip_distance) por borough y hora

Interpretacion: el yield por milla cambia por franja horaria y borough; areas centrales muestran estabilidad y periferia mayor dispersion operacional.

In [21]:
q_r = f"""
SELECT pu_borough, pickup_hour,
       ROUND(AVG(CASE WHEN trip_distance > 0 THEN total_amount / trip_distance END), 2) AS avg_yield_per_mile,
       COUNT(*) AS trips
FROM {obt_table}
GROUP BY 1,2
ORDER BY 1,2
"""
spark.read.format('snowflake').options(**sf_options).option('query', q_r).load().show(200, truncate=False)

+-------------+-----------+------------------+--------+
|PU_BOROUGH   |PICKUP_HOUR|AVG_YIELD_PER_MILE|TRIPS   |
+-------------+-----------+------------------+--------+
|Bronx        |0          |6.22              |130794  |
|Bronx        |1          |6.32              |95888   |
|Bronx        |2          |6.24              |68565   |
|Bronx        |3          |6.03              |59350   |
|Bronx        |4          |5.87              |75368   |
|Bronx        |5          |5.59              |95003   |
|Bronx        |6          |5.16              |162651  |
|Bronx        |7          |5.96              |263456  |
|Bronx        |8          |6.29              |321627  |
|Bronx        |9          |6.22              |297735  |
|Bronx        |10         |6.21              |267612  |
|Bronx        |11         |6.26              |254946  |
|Bronx        |12         |6.27              |259321  |
|Bronx        |13         |6.32              |259715  |
|Bronx        |14         |6.39              |27

## s) Cambios YoY en volumen y ticket promedio por service_type

Interpretacion: se observa caida fuerte de volumen en 2020 y recuperacion posterior; el ticket promedio evoluciona al alza en etapas de recuperacion y cambio de mezcla.

In [22]:
q_s = f"""
WITH base AS (
  SELECT year, service_type, COUNT(*) AS trips, AVG(total_amount) AS avg_ticket
  FROM {obt_table}
  GROUP BY 1,2
), yoy AS (
  SELECT year, service_type, trips, avg_ticket,
         LAG(trips) OVER (PARTITION BY service_type ORDER BY year) AS trips_prev,
         LAG(avg_ticket) OVER (PARTITION BY service_type ORDER BY year) AS avg_ticket_prev
  FROM base
)
SELECT year, service_type, trips, ROUND(avg_ticket, 2) AS avg_ticket,
       ROUND(100.0 * (trips - trips_prev) / NULLIF(trips_prev, 0), 2) AS yoy_trips_pct,
       ROUND(100.0 * (avg_ticket - avg_ticket_prev) / NULLIF(avg_ticket_prev, 0), 2) AS yoy_ticket_pct
FROM yoy
ORDER BY service_type, year
"""
spark.read.format('snowflake').options(**sf_options).option('query', q_s).load().show(50, truncate=False)

+----+------------+---------+----------+-------------+--------------+
|YEAR|SERVICE_TYPE|TRIPS    |AVG_TICKET|YOY_TRIPS_PCT|YOY_TICKET_PCT|
+----+------------+---------+----------+-------------+--------------+
|2015|green       |18704761 |14.81     |NULL         |NULL          |
|2016|green       |15873156 |14.64     |-15.14       |-1.12         |
|2017|green       |11407950 |14.29     |-28.13       |-2.44         |
|2018|green       |8645610  |16.22     |-24.21       |13.54         |
|2019|green       |6033345  |18.36     |-30.21       |13.17         |
|2020|green       |1640298  |20.29     |-72.81       |10.55         |
|2021|green       |1009223  |24.11     |-38.47       |18.82         |
|2022|green       |771009   |18.89     |-23.60       |-21.66        |
|2023|green       |727591   |23.64     |-5.63        |25.13         |
|2024|green       |606480   |24.1      |-16.65       |1.95          |
|2025|green       |544796   |25.13     |-10.17       |4.27          |
|2015|yellow      |1

## t) Dias con alta congestion_surcharge vs dias normales

Interpretacion: dias de alta congestion presentan mayor total_amount promedio que dias normales, coherente con mayor sobrecargo y tiempos de viaje.

In [23]:
q_t = f"""
WITH by_day AS (
  SELECT pickup_date,
         AVG(congestion_surcharge) AS avg_congestion,
         AVG(total_amount) AS avg_total
  FROM {obt_table}
  GROUP BY 1
), tagged AS (
  SELECT *,
         CASE WHEN avg_congestion >= PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY avg_congestion) OVER ()
              THEN 'high_congestion_day'
              ELSE 'normal_day'
         END AS day_type
  FROM by_day
)
SELECT day_type,
       COUNT(*) AS days,
       ROUND(AVG(avg_congestion), 3) AS avg_congestion,
       ROUND(AVG(avg_total), 2) AS avg_total_amount
FROM tagged
GROUP BY 1
ORDER BY 1
"""
spark.read.format('snowflake').options(**sf_options).option('query', q_t).load().show(truncate=False)

+-------------------+----+--------------+----------------+
|DAY_TYPE           |DAYS|AVG_CONGESTION|AVG_TOTAL_AMOUNT|
+-------------------+----+--------------+----------------+
|high_congestion_day|255 |2.337         |23.96           |
|normal_day         |3763|2.225         |20.64           |
+-------------------+----+--------------+----------------+



## Interpretacion ejecutiva (a-t)

a) Las zonas de pickup mas recurrentes se concentran en Manhattan (Upper East Side, Midtown, Union Sq), con estabilidad mensual alta.

b) Los dropoffs top muestran patrones espejo de zonas de alta demanda y corredores de viaje intra-Manhattan.

c) El total_amount crece en boroughs de mayor volumen y el tip_pct promedio es mas alto donde domina pago con tarjeta.

d) El ticket promedio cambia por service_type y mes; yellow mantiene mayor volumen y green muestra ticket medio sensible a estacionalidad.

e) Los picos de viajes se concentran en horas laborales y franjas de salida (manana/tarde), con variacion por dia de semana.

f) Los percentiles p50/p90 de duracion evidencian trayectos mas largos en Queens/Staten Island frente a Manhattan.

g) La velocidad promedio cae en horas pico en zonas centrales; boroughs perifericos sostienen mayores mph por recorridos mas largos y fluidos.

h) Tarjeta concentra mayor participacion y mayor tip_pct; cash mantiene tip_pct cercano a cero.

i) Standard rate concentra mayor monto total por volumen; codigos de aeropuerto muestran mayor distancia promedio.

j) El mix yellow vs green por borough confirma predominio de yellow en Manhattan y mayor peso relativo de green en Brooklyn/Bronx.

k) Los flujos PU->DO top son mayormente intra-zona o entre zonas vecinas de Manhattan, con ticket promedio consistente.

l) passenger_count=1 domina claramente; grupos mayores son minoritarios y muestran ticket medio diferente por tipo de viaje.

m) congestion_surcharge y tolls elevan total_amount en zonas/tramos especificos, especialmente corredores con peajes.

n) Viajes cortos predominan en boroughs densos; viajes largos tienen mayor peso en periferia y muestran componente estacional.

o) Se observan diferencias por vendor en duracion/velocidad promedio, explicables por mix geoespacial y operativo.

p) La propina por hora depende del metodo de pago; tarjeta mantiene niveles estables y superiores en la mayoria de franjas.

q) Zonas con p99 alto de duracion/distancia se asocian a periferia, aeropuertos y posibles eventos de congestion.

r) El yield por milla varia por borough y hora; zonas centrales sostienen yield estable y periferia mayor dispersion.

s) El YoY refleja shocks fuertes (ej. 2020) y recuperacion posterior en volumen y ticket promedio por service_type.

t) En dias de alta congestion, el total_amount promedio es mayor que en dias normales, coherente con sobrecargos y tiempos de viaje.

## Nota metodologica (lectura de negocio)

Para reporting ejecutivo, conviene complementar con una vista `business_clean` excluyendo categorias de baja interpretabilidad (`Unknown`, `N/A`, `EWR`) en borough/zone, sin reemplazar la respuesta oficial del PSet.

In [ ]:
obt_table_business = f"""(
  SELECT *
  FROM {obt_table}
  WHERE COALESCE(pu_borough, 'Unknown') NOT IN ('Unknown', 'N/A', 'EWR')
    AND COALESCE(do_borough, 'Unknown') NOT IN ('Unknown', 'N/A', 'EWR')
    AND COALESCE(pu_zone, 'Unknown') NOT IN ('Unknown', 'N/A')
    AND COALESCE(do_zone, 'Unknown') NOT IN ('Unknown', 'N/A')
)"""

q_business_check = f"""
SELECT COUNT(*) AS rows_business_clean
FROM {obt_table_business}
"""

spark.read.format('snowflake').options(**sf_options).option('query', q_business_check).load().show()